In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail")

silver = spark.table("workspace.retail.silver_online_retail")

canc_lines = silver.filter("invoice_type = 'CANCELLATION'").count()
print("Cancellation lines in Silver (report must match):", f"{canc_lines:,}")

In [0]:
cancellations = (
    silver.filter("invoice_type = 'CANCELLATION'")
    .select(
        F.col("source_row_id").alias("cancel_line_id"),
        F.col("invoice").alias("cancel_invoice"),
        "customer_id", "stock_code", "country",
        F.col("invoice_date").alias("cancel_date"),
        "invoice_month",
        (-F.col("quantity")).alias("returned_qty"),
        F.col("price").alias("unit_price"),
        F.col("line_amount").alias("cancel_amount"),
    )
)

sales = (
    silver.filter("invoice_type = 'SALE' AND customer_id != -1")
    .select("customer_id", "stock_code",
            F.col("invoice_date").alias("sale_date"),
            F.col("quantity").alias("sold_qty"))
)

In [0]:
c, s = cancellations.alias("c"), sales.alias("s")

prior_sold = (
    c.join(
        s,
        (F.col("c.customer_id") == F.col("s.customer_id"))
        & (F.col("c.stock_code") == F.col("s.stock_code"))
        & (F.col("s.sale_date") <= F.col("c.cancel_date")),
        "left",
    )
    .groupBy("c.cancel_line_id")
    .agg(F.coalesce(F.sum("s.sold_qty"), F.lit(0)).alias("prior_sold_qty"))
)

In [0]:
w_prior = (
    Window.partitionBy("customer_id", "stock_code")
    .orderBy("cancel_date", "cancel_line_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)

report = (
    cancellations
    .join(prior_sold, "cancel_line_id", "left")
    .withColumn("prior_returned_qty", F.coalesce(F.sum("returned_qty").over(w_prior), F.lit(0)))
    .withColumn("available_qty", F.col("prior_sold_qty") - F.col("prior_returned_qty"))
    .withColumn("matched_qty",
        F.greatest(F.lit(0), F.least(F.col("returned_qty"), F.col("available_qty"))))
    .withColumn("unmatched_qty", F.col("returned_qty") - F.col("matched_qty"))
    .withColumn("match_status",
        F.when(F.col("customer_id") == -1, "UNMATCHED_NO_CUSTOMER")
         .when(F.col("matched_qty") == F.col("returned_qty"), "FULL")
         .when(F.col("matched_qty") > 0, "PARTIAL")
         .otherwise("UNMATCHED"))
    .withColumn("matched_amount", F.round(F.col("matched_qty") * F.col("unit_price") * -1, 2))
)

(report.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.retail.gold_cancellation_report"))

In [0]:
rep = spark.table("workspace.retail.gold_cancellation_report")

n = rep.count()
print(f"Report rows: {n:,} | Cancellation lines in Silver: {canc_lines:,}")
assert n == canc_lines, "Join lost or duplicated cancellation lines"
assert rep.select("cancel_line_id").distinct().count() == n, "Duplicate cancel_line_id"
assert rep.filter("matched_qty < 0 OR matched_qty > returned_qty").count() == 0
assert rep.filter("matched_qty + unmatched_qty != returned_qty").count() == 0
assert rep.filter("match_status IS NULL").count() == 0

display(
    rep.groupBy("match_status")
    .agg(F.count("*").alias("lines"),
         F.sum("returned_qty").alias("units_returned"),
         F.sum("matched_qty").alias("units_matched"),
         F.round(F.sum("cancel_amount"), 2).alias("cancel_value"))
    .orderBy(F.desc("lines"))
)
print("Cancellation report checks passed")

In [0]:
fact = spark.table("workspace.retail.fact_sales")
dd = spark.table("workspace.retail.dim_date").select("date_key", "year_month")

monthly = (
    fact.join(dd, "date_key", "left")
    .groupBy("year_month")
    .agg(
        F.round(F.sum(F.when(F.col("invoice_type") == "SALE", F.col("line_amount"))), 2).alias("gross_sales"),
        F.round(F.sum(F.when(F.col("invoice_type") == "CANCELLATION", F.col("line_amount"))), 2).alias("cancellations"),
        F.round(F.sum("line_amount"), 2).alias("net_sales"),
        F.round(F.sum(F.when(F.col("dup_seq") > 1, F.col("line_amount"))), 2).alias("possible_duplicate_amount"),
        F.countDistinct("invoice").alias("invoices"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(-F.col("cancellations") / F.col("gross_sales") * 100, 2))
    .orderBy("year_month")
)

monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.gold_monthly_sales")

display(monthly)

In [0]:
m = spark.table("workspace.retail.gold_monthly_sales")
silver_total = silver.agg(F.round(F.sum("line_amount"), 2)).first()[0]
net_total = m.agg(F.round(F.sum("net_sales"), 2)).first()[0]

print("Monthly net total:", net_total, "| Silver total:", silver_total)
assert net_total == silver_total, "Monthly view does not tie to Silver"

gross_plus_canc = m.agg(F.round(F.sum(F.col("gross_sales") + F.coalesce(F.col("cancellations"), F.lit(0))), 2)).first()[0]
assert gross_plus_canc == net_total, "gross + cancellations != net"

assert m.count() == silver.select("invoice_month").distinct().count(), "Month count differs"
print("Monthly reconciliation passed")

In [0]:
rep = spark.table("workspace.retail.gold_cancellation_report")

# Biggest cancellations by value: do a few lines explain the spikes?
display(
    rep.orderBy("cancel_amount")
       .select("invoice_month", "cancel_invoice", "customer_id", "stock_code",
               "returned_qty", "cancel_amount", "match_status")
       .limit(15)
)

# Do UNMATCHED returns cluster early (data starts Dec 2009)?
display(
    rep.filter("match_status = 'UNMATCHED'")
       .groupBy("invoice_month").count().orderBy("invoice_month")
)